## 1. setup
imports, the random seed and the output folder.

In [1]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.decomposition import PCA
import os

SEED = 42

#all results and submissions are saved in this folder
output_folder = "output"
os.makedirs(output_folder, exist_ok=True)

## 2. load data
sentiment data, split 80/20 with the same proportion of each label.

In [2]:
sent_df = pd.read_csv("comp-472-project-fall-2026-concordia/sentiment_train.csv")

X_sent = sent_df.drop(columns=["id", "label"]).to_numpy()
y_sent = sent_df["label"].to_numpy()

In [3]:
X_train, X_val, y_train, y_val = train_test_split(
    X_sent, y_sent,
    test_size=0.2,
    stratify=y_sent,
    random_state=SEED,
)

In [4]:
#image data, same thing as sentiment
img_df = pd.read_csv("comp-472-project-fall-2026-concordia/image_train.csv")

X_img = img_df.drop(columns=["id", "label"]).to_numpy()
y_img = img_df["label"].to_numpy()

In [5]:
X_img_train, X_img_val, y_img_train, y_img_val = train_test_split(
    X_img, y_img,
    test_size=0.2,
    stratify=y_img,
    random_state=SEED,
)

## 3. distance functions
four ways to measure how far a row `x` is from every training row in `X`.

In [6]:
def euclidean_distance(x, X):
    return np.sqrt(((X - x) ** 2).sum(axis=1))

def manhattan_distance(x, X):
    return np.abs(X - x).sum(axis=1)

def minkowski_distance(x, X, p = 3):
    return (np.abs(X - x) ** p).sum(axis=1) ** (1 / p)

def cosine_distance(x, X):
    lengths = np.linalg.norm(X, axis=1) * np.linalg.norm(x)
    #a row of all zeros has length 0, avoid dividing by 0
    lengths[lengths == 0] = 1e-12
    return 1 - (X @ x) / lengths

## 4. feature scaling
each function learns from the training rows only and applies the same change to the other rows.

In [7]:
def no_scaling(X_train, X_other):
    return X_train, X_other

def standardize(X_train, X_other):
    #mean and std come from the training rows only
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0)
    std[std == 0] = 1
    return (X_train - mean) / std, (X_other - mean) / std

def sqrt_scaling(X_train, X_other):
    #for the image histograms: shrinks big counts so rare visual words matter too
    A = np.sqrt(X_train)
    B = np.sqrt(X_other)
    len_A = np.linalg.norm(A, axis=1, keepdims=True)
    len_B = np.linalg.norm(B, axis=1, keepdims=True)
    len_A[len_A == 0] = 1
    len_B[len_B == 0] = 1
    return A / len_A, B / len_B

def pca_whiten(X_train, X_other, n_components=64):
    #squeezes 768 features to 64 and gives each the same spread
    pca = PCA(n_components, whiten=True, random_state=SEED)
    pca.fit(X_train)
    return pca.transform(X_train), pca.transform(X_other)

## 5. knn model
hand-written KNN. `weighted=True` makes closer neighbours count more in the vote.

In [8]:
class KNN:
    def __init__(self, k = 1, distance_fn = euclidean_distance, weighted = False):
        self.k = k
        self.distance_fn = distance_fn
        self.weighted = weighted

    def fit(self, X_train, y_train):
        self.X_train = X_train
        self.y_train= y_train
        return self

    def _vote(self, labels, dists, weighted):
        if weighted:
            #closer neighbours get a bigger vote, add the votes up per label
            weights = 1 / (dists + 1e-9)
            return np.bincount(labels, weights=weights).argmax()
        return Counter(labels).most_common(1)[0][0]

    def _predict(self, x):
        dists = self.distance_fn(x, self.X_train)
        nearest = np.argsort(dists)[:self.k]
        return self._vote(self.y_train[nearest], dists[nearest], self.weighted)

    def predict(self, X):
        prediction = [self._predict(x) for x in X]
        return prediction

    def score(self, X, y):
        preds = self.predict(X)
        return sum(p == t for p, t in zip(preds, y)) / len(y)

    def predict_all_k(self, X, k_values):
        #same as predict, but the distances and sorting are done once per row
        #and reused for every k, instead of redoing them for each k
        #returns two dicts: normal vote and weighted vote
        unweighted_preds = {k: [] for k in k_values}
        weighted_preds = {k: [] for k in k_values}
        for x in X:
            dists = self.distance_fn(x, self.X_train)
            nearest = np.argsort(dists)[:max(k_values)]
            for k in k_values:
                labels = self.y_train[nearest[:k]]
                near_dists = dists[nearest[:k]]
                unweighted_preds[k].append(self._vote(labels, near_dists, False))
                weighted_preds[k].append(self._vote(labels, near_dists, True))
        return unweighted_preds, weighted_preds

## 6. helpers and validation
save-file names that never overwrite, accuracy, and 5-fold cross-validation.

In [9]:
def unique_path(folder, base_name, ext=".csv"):
    path = os.path.join(folder, base_name + ext)
    n = 1
    while os.path.exists(path):
        path = os.path.join(folder, f"{base_name}_{n}{ext}")
        n += 1
    return path

def accuracy(preds, y):
    return np.mean(np.array(preds) == y)

def cross_validate(X, y, k, distance_fn, weighted, scaling_fn, n_folds=5):
    #uses the whole training file and every row is in the validation part exactly once,
    #so the result is less noisy than one split
    folds = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    scores = []
    for train_idx, val_idx in folds.split(X, y):
        X_a, X_b = scaling_fn(X[train_idx], X[val_idx])
        clf = KNN(k, distance_fn=distance_fn, weighted=weighted).fit(X_a, y[train_idx])
        scores.append(clf.score(X_b, y[val_idx]))
    return np.mean(scores), np.std(scores)

## 7. early exploration (sentiment, small validation subset)
quick sweeps on part of the validation set to find the rough range of k.

In [10]:
#old method new method us recursion to find the best K range
# settings: change these, and the file name updates automatically
k_values = [1, 3, 7, 15, 31, 63, 127]
n_val = 500

distance_fns = [euclidean_distance,
                minkowski_distance,  manhattan_distance, cosine_distance]


results = []
for fn in distance_fns:
    for k in k_values:
        clf = KNN(k, distance_fn=fn).fit(X_train, y_train)
        acc = clf.score(X_val[:n_val], y_val[:n_val])
        results.append({"distance": fn.__name__, "k": k, "val_accuracy": acc})
        print(f"{fn.__name__}  k={k}  val acc={acc:.1%}")

best = max(results, key=lambda r: r["val_accuracy"])
best_line = f"best: {best['distance']} k = {best['k']} ({best['val_accuracy']:.1%})"

base_name = f"knn_k({k_values[0]}-{k_values[-1]})_val({n_val})"
save_path = unique_path(output_folder, base_name)
pd.DataFrame(results).to_csv(save_path, index=False)

with open(save_path, "a") as f:
    f.write(best_line + "\n")

print("saved to:", save_path)
print(best_line)

euclidean_distance  k=1  val acc=52.0%
euclidean_distance  k=3  val acc=52.8%
euclidean_distance  k=7  val acc=56.2%
euclidean_distance  k=15  val acc=55.4%
euclidean_distance  k=31  val acc=58.0%
euclidean_distance  k=63  val acc=61.8%
euclidean_distance  k=127  val acc=59.0%
minkowski_distance  k=1  val acc=51.0%
minkowski_distance  k=3  val acc=53.0%
minkowski_distance  k=7  val acc=57.6%
minkowski_distance  k=15  val acc=55.8%
minkowski_distance  k=31  val acc=59.2%
minkowski_distance  k=63  val acc=60.6%
minkowski_distance  k=127  val acc=60.2%
manhattan_distance  k=1  val acc=52.0%
manhattan_distance  k=3  val acc=52.8%
manhattan_distance  k=7  val acc=54.8%
manhattan_distance  k=15  val acc=56.4%
manhattan_distance  k=31  val acc=60.0%
manhattan_distance  k=63  val acc=61.0%
manhattan_distance  k=127  val acc=60.0%
cosine_distance  k=1  val acc=52.8%
cosine_distance  k=3  val acc=53.2%
cosine_distance  k=7  val acc=54.6%
cosine_distance  k=15  val acc=55.2%
cosine_distance  k=31

In [11]:
#best top two winnders
# settings: change these, and the file name updates automatically
k_values = [51, 55,59, 63, 67, 71, 74]
n_val = 1000

distance_fns = [euclidean_distance,
                minkowski_distance,  manhattan_distance]


results = []
for fn in distance_fns:
    for k in k_values:
        clf = KNN(k, distance_fn=fn).fit(X_train, y_train)
        acc = clf.score(X_val[:n_val], y_val[:n_val])
        results.append({"distance": fn.__name__, "k": k, "val_accuracy": acc})
        print(f"{fn.__name__}  k={k}  val acc={acc:.1%}")

best = max(results, key=lambda r: r["val_accuracy"])
best_line = f"best: {best['distance']} k = {best['k']} ({best['val_accuracy']:.1%})"

base_name = f"knn_k_top_3_winner({k_values[0]}-{k_values[-1]})_val({n_val})"
save_path = unique_path(output_folder, base_name)
pd.DataFrame(results).to_csv(save_path, index=False)

with open(save_path, "a") as f:
    f.write(best_line + "\n")

print("saved to:", save_path)
print(best_line)

euclidean_distance  k=51  val acc=59.1%
euclidean_distance  k=55  val acc=60.1%
euclidean_distance  k=59  val acc=60.2%
euclidean_distance  k=63  val acc=60.5%
euclidean_distance  k=67  val acc=59.6%
euclidean_distance  k=71  val acc=59.2%
euclidean_distance  k=74  val acc=59.5%
minkowski_distance  k=51  val acc=59.8%
minkowski_distance  k=55  val acc=59.4%
minkowski_distance  k=59  val acc=59.8%
minkowski_distance  k=63  val acc=59.4%
minkowski_distance  k=67  val acc=59.5%
minkowski_distance  k=71  val acc=58.6%
minkowski_distance  k=74  val acc=58.6%
manhattan_distance  k=51  val acc=59.8%
manhattan_distance  k=55  val acc=59.7%
manhattan_distance  k=59  val acc=59.4%
manhattan_distance  k=63  val acc=59.9%
manhattan_distance  k=67  val acc=59.7%
manhattan_distance  k=71  val acc=59.3%
manhattan_distance  k=74  val acc=59.8%
saved to: output/knn_k_top_3_winner(51-74)_val(1000)_2.csv
best: euclidean_distance k = 63 (60.5%)


## 8. full sweep: sentiment
every scaling, distance, voting type and k on the whole validation part.

In [12]:
#full sentiment validation set, every scaling x distance, both voting types, k 1-301
#uses predict_all_k so the distances are only computed once per row
k_values = list(range(1, 302, 2))
n_val = len(X_val)

distance_fns = [euclidean_distance,
                minkowski_distance,  manhattan_distance, cosine_distance]
scaling_list = [("none", no_scaling), ("standardize", standardize), ("pca_whiten", pca_whiten)]

sent_results = []
for scaling_name, scaling_fn in scaling_list:
    #the scaling is learned on the train part only
    X_a, X_b = scaling_fn(X_train, X_val)
    for fn in distance_fns:
        clf = KNN(distance_fn=fn).fit(X_a, y_train)
        unweighted_preds, weighted_preds = clf.predict_all_k(X_b, k_values)
        run = []
        for k in k_values:
            run.append({"scaling": scaling_name, "distance": fn.__name__, "voting": "majority",
                        "k": k, "val_accuracy": accuracy(unweighted_preds[k], y_val)})
            run.append({"scaling": scaling_name, "distance": fn.__name__, "voting": "weighted",
                        "k": k, "val_accuracy": accuracy(weighted_preds[k], y_val)})
        sent_results.extend(run)
        run_best = max(run, key=lambda r: r["val_accuracy"])
        print(f"{scaling_name}  {fn.__name__}  best: {run_best['voting']} k={run_best['k']}  val acc={run_best['val_accuracy']:.1%}")

best = max(sent_results, key=lambda r: r["val_accuracy"])
best_line = f"best: {best['scaling']} {best['distance']} {best['voting']} k = {best['k']} ({best['val_accuracy']:.1%})"

base_name = f"knn_sentiment_grid_k({k_values[0]}-{k_values[-1]})_val({n_val})"
save_path = unique_path(output_folder, base_name)
pd.DataFrame(sent_results).to_csv(save_path, index=False)

with open(save_path, "a") as f:
    f.write(best_line + "\n")

sent_best = best
print("saved to:", save_path)
print(best_line)

none  euclidean_distance  best: weighted k=79  val acc=59.8%
none  minkowski_distance  best: weighted k=33  val acc=58.5%
none  manhattan_distance  best: majority k=43  val acc=59.0%
none  cosine_distance  best: weighted k=63  val acc=58.8%
standardize  euclidean_distance  best: majority k=61  val acc=59.1%
standardize  minkowski_distance  best: weighted k=53  val acc=58.9%
standardize  manhattan_distance  best: majority k=45  val acc=59.0%
standardize  cosine_distance  best: weighted k=87  val acc=58.8%
pca_whiten  euclidean_distance  best: weighted k=105  val acc=60.6%
pca_whiten  minkowski_distance  best: majority k=153  val acc=60.3%
pca_whiten  manhattan_distance  best: majority k=121  val acc=60.2%
pca_whiten  cosine_distance  best: majority k=283  val acc=62.0%
saved to: output/knn_sentiment_grid_k(1-301)_val(2400)_1.csv
best: pca_whiten cosine_distance majority k = 283 (62.0%)


## 9. full sweep: image
same as above for the image data.

In [13]:
#full image validation set, every scaling x distance, both voting types, k 1-201
#uses predict_all_k so the distances are only computed once per row
k_values = list(range(1, 202, 2))
n_val = len(X_img_val)

distance_fns = [euclidean_distance,
                minkowski_distance,  manhattan_distance, cosine_distance]
scaling_list = [("none", no_scaling), ("standardize", standardize), ("sqrt", sqrt_scaling)]

img_results = []
for scaling_name, scaling_fn in scaling_list:
    #the scaling is learned on the train part only
    X_a, X_b = scaling_fn(X_img_train, X_img_val)
    for fn in distance_fns:
        clf = KNN(distance_fn=fn).fit(X_a, y_img_train)
        unweighted_preds, weighted_preds = clf.predict_all_k(X_b, k_values)
        run = []
        for k in k_values:
            run.append({"scaling": scaling_name, "distance": fn.__name__, "voting": "majority",
                        "k": k, "val_accuracy": accuracy(unweighted_preds[k], y_img_val)})
            run.append({"scaling": scaling_name, "distance": fn.__name__, "voting": "weighted",
                        "k": k, "val_accuracy": accuracy(weighted_preds[k], y_img_val)})
        img_results.extend(run)
        run_best = max(run, key=lambda r: r["val_accuracy"])
        print(f"{scaling_name}  {fn.__name__}  best: {run_best['voting']} k={run_best['k']}  val acc={run_best['val_accuracy']:.1%}")

best = max(img_results, key=lambda r: r["val_accuracy"])
best_line = f"best: {best['scaling']} {best['distance']} {best['voting']} k = {best['k']} ({best['val_accuracy']:.1%})"

base_name = f"knn_image_grid_k({k_values[0]}-{k_values[-1]})_val({n_val})"
save_path = unique_path(output_folder, base_name)
pd.DataFrame(img_results).to_csv(save_path, index=False)

with open(save_path, "a") as f:
    f.write(best_line + "\n")

img_best = best
print("saved to:", save_path)
print(best_line)

none  euclidean_distance  best: majority k=1  val acc=53.1%
none  minkowski_distance  best: majority k=1  val acc=41.6%
none  manhattan_distance  best: majority k=1  val acc=39.3%
none  cosine_distance  best: majority k=1  val acc=53.1%
standardize  euclidean_distance  best: majority k=1  val acc=43.4%
standardize  minkowski_distance  best: majority k=1  val acc=39.7%
standardize  manhattan_distance  best: majority k=1  val acc=30.9%
standardize  cosine_distance  best: majority k=1  val acc=53.4%
sqrt  euclidean_distance  best: majority k=1  val acc=57.2%
sqrt  minkowski_distance  best: majority k=1  val acc=40.3%
sqrt  manhattan_distance  best: majority k=1  val acc=44.2%
sqrt  cosine_distance  best: majority k=1  val acc=57.2%
saved to: output/knn_image_grid_k(1-201)_val(4000)_1.csv
best: sqrt euclidean_distance majority k = 1 (57.2%)


## 10. cross-validation of the top settings
the 3 best settings of each dataset are checked on the whole training file, the best one is kept.

In [14]:
#take the 3 best different settings from each grid (using each one's best k)
#and check them with 5-fold cross-validation on the whole training file
distance_by_name = {fn.__name__: fn for fn in distance_fns}
scaling_by_name = {"none": no_scaling, "standardize": standardize, "sqrt": sqrt_scaling, "pca_whiten": pca_whiten}

def top_settings(results, how_many = 3):
    ordered = sorted(results, key=lambda r: r["val_accuracy"], reverse=True)
    top = []
    seen = []
    for r in ordered:
        combo = (r["scaling"], r["distance"], r["voting"])
        if combo not in seen:
            seen.append(combo)
            top.append(r)
        if len(top) == how_many:
            break
    return top

def run_cv(dataset_name, X, y, results):
    cv_rows = []
    for r in top_settings(results):
        mean, std = cross_validate(X, y, r["k"], distance_by_name[r["distance"]],
                                   r["voting"] == "weighted", scaling_by_name[r["scaling"]])
        cv_rows.append({"dataset": dataset_name, "scaling": r["scaling"], "distance": r["distance"],
                        "voting": r["voting"], "k": r["k"], "val_accuracy": r["val_accuracy"],
                        "cv_mean": mean, "cv_std": std})
        print(f"{dataset_name}  {r['scaling']} {r['distance']} {r['voting']} k={r['k']}  cv acc={mean:.1%} +- {std:.1%}")
    return cv_rows

sent_cv = run_cv("sentiment", X_sent, y_sent, sent_results)
img_cv = run_cv("image", X_img, y_img, img_results)

#the setting with the best cross-validation mean is the final one
sent_final = max(sent_cv, key=lambda r: r["cv_mean"])
img_final = max(img_cv, key=lambda r: r["cv_mean"])

save_path = unique_path(output_folder, "knn_cross_validation")
pd.DataFrame(sent_cv + img_cv).to_csv(save_path, index=False)

print("saved to:", save_path)
print(f"sentiment final: {sent_final['scaling']} {sent_final['distance']} {sent_final['voting']} k = {sent_final['k']}")
print(f"image final: {img_final['scaling']} {img_final['distance']} {img_final['voting']} k = {img_final['k']}")

sentiment  pca_whiten cosine_distance majority k=283  cv acc=62.2% +- 0.7%
sentiment  pca_whiten cosine_distance weighted k=285  cv acc=62.3% +- 0.7%
sentiment  pca_whiten euclidean_distance weighted k=105  cv acc=59.5% +- 0.7%
image  sqrt euclidean_distance majority k=1  cv acc=57.4% +- 0.9%
image  sqrt euclidean_distance weighted k=1  cv acc=57.4% +- 0.9%
image  sqrt cosine_distance majority k=1  cv acc=57.4% +- 0.9%
saved to: output/knn_cross_validation_1.csv
sentiment final: pca_whiten cosine_distance weighted k = 285
image final: sqrt euclidean_distance majority k = 1


## 11. error check on validation
which classes get confused, and whether the predicted class counts look right.

In [15]:
#refit the final settings on the train part, look at where the validation mistakes are
def error_check(dataset_name, final, X_a, X_b, y_a, y_b, show_classes = None):
    X_a, X_b = scaling_by_name[final["scaling"]](X_a, X_b)
    clf = KNN(final["k"], distance_fn=distance_by_name[final["distance"]],
              weighted=(final["voting"] == "weighted")).fit(X_a, y_a)
    preds = np.array(clf.predict(X_b))
    print(dataset_name, "validation accuracy:", accuracy(preds, y_b))

    #accuracy of each class
    class_acc = {}
    for c in np.unique(y_b):
        class_acc[c] = np.mean(preds[y_b == c] == c)

    if show_classes is None:
        print(pd.crosstab(y_b, preds, rownames=["true"], colnames=["predicted"]))
        for c in class_acc:
            print(f"class {c} accuracy: {class_acc[c]:.1%}")
    else:
        #many classes: a full table is too wide, only show the worst ones
        worst = sorted(class_acc, key=lambda c: class_acc[c])[:show_classes]
        print(f"{show_classes} worst classes:")
        for c in worst:
            print(f"class {c} accuracy: {class_acc[c]:.1%}")

    #predicted class counts vs true counts (sanity check, they should look alike)
    #(printed as lists, one number per class 0, 1, 2, ...)
    print("predicted counts:", np.bincount(preds).tolist())
    print("true counts:", np.bincount(y_b).tolist())

error_check("sentiment", sent_final, X_train, X_val, y_train, y_val)
error_check("image", img_final, X_img_train, X_img_val, y_img_train, y_img_val, show_classes=5)

sentiment validation accuracy: 0.6191666666666666
predicted    0    1    2
true                    
0          584  123   93
1          234  400  166
2          109  189  502
class 0 accuracy: 73.0%
class 1 accuracy: 50.0%
class 2 accuracy: 62.7%
predicted counts: [927, 712, 761]
true counts: [800, 800, 800]
image validation accuracy: 0.57175
5 worst classes:
class 29 accuracy: 40.0%
class 22 accuracy: 42.5%
class 9 accuracy: 43.8%
class 41 accuracy: 45.0%
class 6 accuracy: 46.2%
predicted counts: [83, 72, 79, 77, 78, 90, 72, 101, 101, 79, 82, 74, 113, 101, 85, 83, 77, 59, 78, 87, 70, 76, 60, 60, 67, 105, 75, 77, 71, 67, 64, 73, 83, 52, 74, 112, 77, 98, 108, 83, 60, 61, 60, 62, 70, 84, 102, 91, 97, 90]
true counts: [80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80, 80]


## 12. final model and kaggle submission
refit on all training rows with the final settings, predict the test set and write the file.

In [16]:
#final model: best setting for each dataset, using all the training rows, predicting the kaggle test set
sent_test = pd.read_csv("comp-472-project-fall-2026-concordia/sentiment_test.csv")
img_test = pd.read_csv("comp-472-project-fall-2026-concordia/image_test.csv")

#the scaling is learned on all the training rows, then applied to the test rows
X_a, X_b = scaling_by_name[sent_final["scaling"]](X_sent, sent_test.drop(columns=["id"]).to_numpy())
sent_model = KNN(sent_final["k"], distance_fn=distance_by_name[sent_final["distance"]],
                 weighted=(sent_final["voting"] == "weighted")).fit(X_a, y_sent)
sentiment_predictions = sent_model.predict(X_b)

X_a, X_b = scaling_by_name[img_final["scaling"]](X_img, img_test.drop(columns=["id"]).to_numpy())
img_model = KNN(img_final["k"], distance_fn=distance_by_name[img_final["distance"]],
                weighted=(img_final["voting"] == "weighted")).fit(X_a, y_img)
image_predictions = img_model.predict(X_b)

#same as the code on the kaggle page
sample = pd.read_csv("comp-472-project-fall-2026-concordia/sample_submission.csv")

image_results = pd.DataFrame({"id": img_test["id"], "label": image_predictions})
sentiment_results = pd.DataFrame({"id": sent_test["id"], "label": sentiment_predictions})
predictions = pd.concat([image_results, sentiment_results], ignore_index=True)

submission = sample[["id"]].merge(predictions, on="id", how="left", validate="one_to_one")
assert len(submission) == len(sample)
assert submission["label"].notna().all()
submission["label"] = submission["label"].astype(int)

save_path = unique_path(output_folder, "submission_knn")
submission.to_csv(save_path, index=False)

print("saved to:", save_path)
print("sentiment predicted label counts:", np.bincount(sentiment_predictions).tolist())
print(f"sentiment: {sent_final['scaling']} {sent_final['distance']} {sent_final['voting']} k = {sent_final['k']} ({sent_final['cv_mean']:.1%} cross-validation)")
print(f"image: {img_final['scaling']} {img_final['distance']} {img_final['voting']} k = {img_final['k']} ({img_final['cv_mean']:.1%} cross-validation)")

saved to: output/submission_knn_1.csv
sentiment predicted label counts: [2581, 1031, 1388]
sentiment: pca_whiten cosine_distance weighted k = 285 (62.3% cross-validation)
image: sqrt euclidean_distance majority k = 1 (57.4% cross-validation)
